# PVGIS-only ST-GNN + SDE-Net pipeline — t+1h

Orchestratore riproducibile del run paper-faithful originale sul server **newzealand**. Addestra un solo modello con target t+1h e valuta separatamente campioni normali e anomali usando le decisioni puntuali MTGFlow sulla coppia `(location, timestamp target)`.

Il notebook non duplica training o report logic: costruisce i comandi e legge gli output tramite `physiq_pv.experiments.sde_pipeline`.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
print('repo root:', ROOT)

## 1. Dati e label MTGFlow

In [ ]:
PVGIS_DIR = pipe.PVGIS_DIR
ANOMALY_SOURCE = 'detector'
DETECTOR = 'mtgflow'
DETECTOR_SEED = 15
DETECTOR_ROOT = Path('outputs/pvgis_mtgflow/downstream_dense') / f'seed_{DETECTOR_SEED}'
TEST_ANOMALY_SCORES = str(DETECTOR_ROOT / 'anomaly_scores.csv')
TRAIN_ANOMALY_SCORES = str(DETECTOR_ROOT / 'train_anomaly_scores.csv')

checks = {
    'PVGIS dir': Path(PVGIS_DIR).is_dir(),
    'MTGFlow test scores': Path(TEST_ANOMALY_SCORES).is_file(),
    'MTGFlow train scores': Path(TRAIN_ANOMALY_SCORES).is_file(),
}
for name, ok in checks.items():
    print(('OK  ' if ok else 'MISS'), name)
if not all(checks.values()):
    raise FileNotFoundError('Creare prima i CSV MTGFlow densi richiesti.')

## 2. Configurazione originale t+1h

In [ ]:
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'horizon': 1,
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False,
    'anomaly_source': ANOMALY_SOURCE,
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1,
    'detector_regional_quantile': 0.975,
}
assert BASE_CONFIG['horizon'] == 1
out_dir = pipe.make_out_dir(BASE_CONFIG)
run_name = pipe.make_run_name(BASE_CONFIG)
print('config:', BASE_CONFIG)
print('out_dir:', out_dir)
print('run_name:', run_name)

In [ ]:
TRAIN_COMMAND = pipe.build_train_command(
    BASE_CONFIG, out_dir=out_dir, run_name=run_name,
    pvgis_dir=PVGIS_DIR, test_anomaly_scores=TEST_ANOMALY_SCORES,
    train_anomaly_scores=TRAIN_ANOMALY_SCORES, device='cuda', use_wandb=True,
)
assert TRAIN_COMMAND[TRAIN_COMMAND.index('--horizon') + 1] == '1'
print(' \\n  '.join(TRAIN_COMMAND))

## 3. Training oppure relabel evaluation-only

In [ ]:
RUN_TRAINING = True
EVAL_ONLY_PREDICTIONS = None
ALLOW_OVERWRITE = False

if RUN_TRAINING and EVAL_ONLY_PREDICTIONS:
    raise ValueError('Scegliere training oppure evaluation-only, non entrambi.')
if RUN_TRAINING:
    pipe.ensure_output_dir_available(out_dir, allow_overwrite=ALLOW_OVERWRITE)
    subprocess.run(TRAIN_COMMAND, check=True, cwd=ROOT)
elif EVAL_ONLY_PREDICTIONS:
    paths = pipe.relabel_detector_predictions_file(
        EVAL_ONLY_PREDICTIONS, TEST_ANOMALY_SCORES, TRAIN_ANOMALY_SCORES,
        out_dir,
        regional_quantile=BASE_CONFIG['detector_regional_quantile'],
        min_temporal_coverage=BASE_CONFIG['detector_min_temporal_coverage'],
        allow_overwrite=ALLOW_OVERWRITE,
    )
    print('Evaluation-only outputs:', paths)
else:
    print('Training ed evaluation-only disattivati.')

## 4. Analisi post-hoc t+1h

In [ ]:
RUN_ANALYSIS = True
predictions_path = Path(out_dir) / 'predictions.csv'
ANALYSIS_COMMAND = pipe.build_analysis_command(out_dir, BASE_CONFIG)
if RUN_ANALYSIS:
    if not predictions_path.is_file():
        raise FileNotFoundError(predictions_path)
    subprocess.run(ANALYSIS_COMMAND, check=True, cwd=ROOT)
else:
    print('RUN_ANALYSIS=False: analisi non avviata.')

In [ ]:
RESULT_FILES = [
    'metrics_global.csv', 'metrics_by_anomaly_label.csv', 'metrics_daytime.csv',
    'daytime_bin_summary.csv', 'daytime_bin_anomaly_metrics.csv',
    'frequency_weighted_bin_summary.csv', 'uncertainty_response.csv',
    'sharpness_overview.csv',
]
RESULTS = {}
for name in RESULT_FILES:
    path = Path(out_dir) / name
    RESULTS[name] = pd.read_csv(path) if path.is_file() else None
    print(('OK  ' if RESULTS[name] is not None else '--  ') + name)
display(RESULTS['metrics_by_anomaly_label.csv'])

In [ ]:
FIGURE_PATHS = pipe.build_posthoc_figures(out_dir)
print({name: str(path) for name, path in FIGURE_PATHS.items()})

## Interpretazione

Tabelle e figure descrivono esclusivamente la previsione t+1h. La separazione normale/anomalo usa `anomaly_group`, ottenuto dalla decisione MTGFlow sullo stesso target `(location, timestamp)`; `event_group` non sostituisce questa label puntuale.